# Deepfake Audio-Video Detection Pipeline Exploration

This notebook demonstrates the underlying features and models used in the Deepfake Detection System. It covers:
1. Loading an input video file using OpenCV and extracting a target frame.
2. Generating visual frame embeddings using a pretrained **EfficientNet-B0** model from `timm`.
3. Extracting and loading the audio channel using FFmpeg and `librosa`.
4. Generating audio embeddings using **Wav2Vec2** from Hugging Face `transformers`.

## 1. Setup and Package Imports

First, we make sure that our environment is configured and import the necessary libraries.

In [ ]:
import os
import cv2
import torch
import timm
import librosa
import numpy as np
import matplotlib.pyplot as plt
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# Check device availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 2. Load Models

We load `efficientnet_b0` for visual feature extraction and `wav2vec2-base-960h` for speech/audio feature extraction.

In [ ]:
# Load EfficientNet-B0 (feature extractor with no classifier head)
visual_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0).to(device).eval()
print("Loaded EfficientNet-B0 model successfully.")

# Load Wav2Vec2 processor and model
audio_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
audio_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device).eval()
print("Loaded Wav2Vec2 model and processor successfully.")

## 3. Visual Feature Extraction

We extract the first frame from the video, resize it to 224x224 (required size), convert it to a PyTorch tensor, and run it through the visual model.

In [ ]:
def extract_and_show_frame(video_path):
    if not os.path.exists(video_path):
        print(f"Error: Video path '{video_path}' does not exist. Please place a sample video there.")
        return None

    # Open video capture
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print("Failed to read frame.")
        return None
        
    # Preprocessing
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))
    
    # Display frame
    plt.imshow(img_resized)
    plt.title("Extracted & Preprocessed Video Frame")
    plt.axis('off')
    plt.show()
    
    # Extract features
    tensor = torch.tensor(img_resized / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)
    with torch.no_grad():
        v_emb = visual_model(tensor).cpu().numpy()
    
    print(f"Visual Embedding shape: {v_emb.shape}")
    return v_emb

# Run extraction on a placeholder path (replace with actual path to test)
visual_emb = extract_and_show_frame("../samples/Students_Coding_Video_Generated.mp4")

## 4. Audio Feature Extraction

Next, we extract the audio track from the video using FFmpeg and preprocess it into Wav2Vec2 embeddings.

In [ ]:
def extract_audio_embeddings(video_path):
    if not os.path.exists(video_path):
        print("Video file not found.")
        return None
        
    os.makedirs("../temp", exist_ok=True)
    wav_path = "../temp/temp_audio.wav"
    
    # Run FFmpeg to convert video to WAV
    command = [
        "ffmpeg", "-y", "-i", video_path, 
        "-vn", "-ac", "1", "-ar", "16000", wav_path
    ]
    
    import subprocess
    try:
        subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("FFmpeg run failed. Please ensure FFmpeg is installed and added to PATH.")
        return None
        
    # Load signal
    wav, sr = librosa.load(wav_path, sr=16000)
    print(f"Loaded audio file. Sample rate: {sr} Hz, Audio shape: {wav.shape}")
    
    # Clean up temp file
    if os.path.exists(wav_path):
        os.remove(wav_path)
        
    # Get embeddings
    inputs = audio_processor(wav, sampling_rate=sr, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = audio_model(**{k: v.to(device) for k, v in inputs.items()})
    
    a_emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    print(f"Audio Embedding shape: {a_emb.shape}")
    return a_emb

audio_emb = extract_audio_embeddings("../samples/Students_Coding_Video_Generated.mp4")

## 5. Summary and Multi-Modal Fusion Idea

To build a full deepfake detection model on top of these features, you can concatenate the visual embedding `(1, 1280)` and audio embedding `(1, 768)` to create a multi-modal feature vector `(1, 2048)`. 

This fused embedding can then be passed to a classifier (such as a Logistic Regression classifier, Multi-Layer Perceptron (MLP), or a recurrent model like LSTM/GRU over multiple video frames) to yield the final deepfake classification probability.